# Evaluación Parcial N°1 — Fundamentos de Deep Learning
## Clasificación de imágenes CIFAR-10 con Perceptrón Multicapa (MLP)

**Curso:** DLY0100 — Deep Learning
**Integrantes:** *(completar nombres)*
**Fecha:** *(completar)*

---

## 1. Introducción

### 1.1 Descripción del problema

El presente trabajo utiliza el dataset **CIFAR-10** (Krizhevsky, 2009), compuesto por 60.000 imágenes en color de 32×32 píxeles, distribuidas en 10 clases mutuamente excluyentes (avión, auto, pájaro, gato, ciervo, perro, rana, caballo, barco, camión), con 6.000 imágenes por clase (5.000 en el conjunto de entrenamiento original y 1.000 en el de test).

El objetivo es implementar un **Perceptrón Multicapa (MLP)** — es decir, una red *feed-forward* totalmente conectada, sin capas convolucionales — capaz de clasificar cada imagen en una de las 10 clases a partir de sus valores de píxel aplanados en un vector de 3.072 valores (32×32×3 canales RGB).

### 1.2 Objetivo del modelo

El objetivo de este trabajo es implementar, entrenar y evaluar una red neuronal artificial (MLP) utilizando **TensorFlow/Keras**, aplicando:

- Carga y preprocesamiento adecuado de los datos de imagen.
- Configuración y análisis de hiperparámetros clave (épocas, tasa de aprendizaje, tamaño de batch).
- Comparación de funciones de activación y de error.
- Técnicas de optimización y regularización (Dropout, L2, Early Stopping).
- Evaluación mediante métricas estándar de clasificación (accuracy, precision, recall, F1-score).
- Análisis crítico de los resultados obtenidos y justificación de las decisiones tomadas.

### 1.3 Estructura del notebook

1. Configuración inicial e importación de librerías.
2. Carga y exploración de los datos de CIFAR-10.
3. Preprocesamiento de las imágenes.
4. Definición de la arquitectura del modelo (MLP).
5. Entrenamiento del modelo base.
6. Análisis del efecto de los hiperparámetros (tasa de aprendizaje, batch size, capacidad de la red).
7. Comparación de funciones de activación.
8. Comparación de funciones de error (loss).
9. Técnicas de optimización y regularización (Dropout, L2, Early Stopping).
10. Ajuste final de hiperparámetros y reentrenamiento.
11. Evaluación del modelo con métricas de clasificación.
12. Comparación de configuraciones y tabla resumen.
13. Conclusiones generales.
14. Referencias y anexo de herramientas utilizadas.

**Nota metodológica:** un MLP no explota la estructura espacial de una imagen (a diferencia de una red convolucional), por lo que su techo de desempeño en CIFAR-10 es limitado — esto se retoma en las conclusiones.


## 2. Configuración inicial

Importamos en un único bloque todas las librerías que se utilizarán a lo largo del notebook:

- **NumPy / Pandas**: manejo de arreglos numéricos (imágenes) y de tablas resumen.
- **scikit-learn**: partición de datos (`train_test_split`) y métricas de evaluación (`accuracy_score`, `precision_recall_fscore_support`, `confusion_matrix`, `classification_report`).
- **TensorFlow / Keras**: construcción, entrenamiento y regularización de la red neuronal (MLP), además de la carga del dataset CIFAR-10 (`keras.datasets.cifar10`).
- **Plotly**: todas las visualizaciones del notebook (gráficos interactivos con etiquetas, leyenda y tooltips detallados), tal como lo exige la pauta del trabajo.
- **IPython.display**: para mostrar tablas (`display`) de forma clara y separada de los mensajes de texto (`print`).

Además fijamos una semilla (`SEED`) para que los resultados sean reproducibles.


In [ ]:
# ============================================================
# Importación de librerías (TODOS los imports del notebook van aquí)
# ============================================================

# --- Manejo de datos ---
import numpy as np
import pandas as pd

# --- Preprocesamiento y métricas (scikit-learn) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

# --- Deep Learning (TensorFlow / Keras) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# --- Visualización interactiva (Plotly, requerido por la pauta del trabajo) ---
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# --- Utilidades para mostrar texto y tablas por separado ---
from IPython.display import display

# ============================================================
# Reproducibilidad
# ============================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Librerías importadas correctamente.")
print(f"Versión de TensorFlow: {tf.__version__}")


**Nota:** fijar la semilla (`SEED`) en NumPy y TensorFlow permite que, en la medida de lo posible, los resultados de los experimentos (pesos iniciales, muestreo del subconjunto de datos, orden de batches) sean reproducibles entre distintas ejecuciones del notebook.


## 3. Carga y exploración de los datos

### 3.1 Carga del dataset CIFAR-10

Se utiliza `keras.datasets.cifar10.load_data()`, que descarga y entrega directamente los arreglos de imágenes (`uint8`, forma `(N, 32, 32, 3)`) y de etiquetas (enteros 0-9) tanto para el conjunto de entrenamiento original (50.000 imágenes) como para el de test original (10.000 imágenes).


In [ ]:
# ============================================================
# Carga del dataset CIFAR-10
# ============================================================
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

# Las etiquetas vienen con forma (N, 1); las dejamos como (N,)
y_train_full = y_train_full.flatten()
y_test_full = y_test_full.flatten()

CLASS_NAMES = ["avión", "auto", "pájaro", "gato", "ciervo",
               "perro", "rana", "caballo", "barco", "camión"]
N_CLASSES = len(CLASS_NAMES)

print(f"Entrenamiento original: {x_train_full.shape}, etiquetas: {y_train_full.shape}")
print(f"Test original:          {x_test_full.shape}, etiquetas: {y_test_full.shape}")
print(f"Clases ({N_CLASSES}): {CLASS_NAMES}")


**Hallazgos:** *(Completar: confirmar que las dimensiones cargadas coinciden con lo documentado por los autores del dataset — 50.000/10.000 imágenes de 32×32×3 — y que las 10 clases quedaron correctamente etiquetadas.)*


### 3.2 Muestra visual de imágenes por clase

Antes de reducir el tamaño del dataset, se inspeccionan visualmente ejemplos de cada clase para verificar que la carga y el etiquetado son correctos.


In [ ]:
# ============================================================
# Grilla de ejemplos, una imagen por clase (Plotly)
# ============================================================
fig_ejemplos = make_subplots(rows=2, cols=5, subplot_titles=CLASS_NAMES)

for clase in range(N_CLASSES):
    idx_ejemplo = np.where(y_train_full == clase)[0][0]
    fila, columna = divmod(clase, 5)
    fig_ejemplos.add_trace(
        go.Image(z=x_train_full[idx_ejemplo]),
        row=fila + 1, col=columna + 1,
    )

fig_ejemplos.update_xaxes(showticklabels=False)
fig_ejemplos.update_yaxes(showticklabels=False)
fig_ejemplos.update_layout(
    title="Un ejemplo por clase — CIFAR-10",
    template="plotly_white",
    showlegend=False,
    height=420,
)
fig_ejemplos.show()


**Hallazgos:** *(Completar: confirmar visualmente que cada imagen corresponde efectivamente a la etiqueta indicada, y comentar la baja resolución —32×32— como un factor que hace más difícil la tarea de clasificación, especialmente para un MLP.)*


### 3.3 Distribución de clases

Se verifica la distribución de clases del conjunto de entrenamiento original. Por diseño, CIFAR-10 está perfectamente balanceado (5.000 imágenes por clase en entrenamiento), lo que se confirma a continuación.


In [ ]:
# ============================================================
# Distribución de la variable objetivo (Plotly)
# ============================================================
conteo_clases = pd.Series(y_train_full).value_counts().sort_index()

fig_clases = go.Figure(
    data=[
        go.Bar(
            x=[CLASS_NAMES[i] for i in conteo_clases.index],
            y=conteo_clases.values,
            text=conteo_clases.values,
            textposition="outside",
            marker_color="#4C78A8",
            hovertemplate="Clase: %{x}<br>Cantidad: %{y}<extra></extra>",
            name="Frecuencia",
        )
    ]
)
fig_clases.update_layout(
    title="Distribución de clases — conjunto de entrenamiento original",
    xaxis_title="Clase",
    yaxis_title="Cantidad de imágenes",
    legend_title_text="Serie",
    template="plotly_white",
)
fig_clases.show()


**Hallazgos:** *(Completar: confirmar que las 10 clases están balanceadas, y señalar por qué esto simplifica la interpretación de accuracy como métrica —aunque igualmente se reportan precision/recall/F1 por clase más adelante.)*


## 4. Preprocesamiento de los datos

### 4.1 Reducción del dataset (muestreo estratificado)

Entrenar un MLP sobre las 50.000 imágenes de entrenamiento, repitiendo el proceso para **cada** experimento de hiperparámetros de este notebook (variaciones de tasa de aprendizaje, batch size, arquitectura, activación, regularización, etc.), implicaría un tiempo de ejecución poco práctico en un entorno sin GPU. Por esta razón, y siguiendo el mismo criterio usado en los notebooks de laboratorio del curso, se trabaja con un **subconjunto reducido y estratificado** del dataset:

- **Entrenamiento:** `N_TRAIN` = 9.000 imágenes (900 por clase).
- **Validación:** `N_VAL` = 4.000 imágenes (400 por clase) — usada para monitorear el error de generalización durante el entrenamiento y aplicar Early Stopping.
- **Test:** `N_TEST` = 2.000 imágenes (200 por clase) — **nunca** se utiliza durante el entrenamiento ni en la selección de hiperparámetros; se reserva exclusivamente para la evaluación final.

El muestreo es **estratificado por clase** (no aleatorio simple) para garantizar que el subconjunto reducido conserve el balance de clases del dataset original. Estos valores son ajustables mediante las constantes definidas a continuación si el tiempo de ejecución disponible es distinto.


In [ ]:
# ============================================================
# Función de muestreo estratificado
# ============================================================
def muestrear_estratificado(X, y, n_total, n_clases, seed):
    """Selecciona n_total muestras de X/y manteniendo la misma cantidad
    de ejemplos por clase (muestreo estratificado), para preservar el
    balance de clases del dataset original en el subconjunto reducido."""
    n_por_clase = n_total // n_clases
    rng = np.random.RandomState(seed)
    indices = []
    for clase in range(n_clases):
        idx_clase = np.where(y == clase)[0]
        elegidos = rng.choice(idx_clase, size=n_por_clase, replace=False)
        indices.append(elegidos)
    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return X[indices], y[indices]


# ============================================================
# Tamaños del subconjunto reducido (ajustables)
# ============================================================
N_TRAIN = 9000
N_VAL = 4000
N_TEST = 2000

x_trainval, y_trainval = muestrear_estratificado(
    x_train_full, y_train_full, N_TRAIN + N_VAL, N_CLASSES, SEED
)
x_train, x_val, y_train, y_val = train_test_split(
    x_trainval, y_trainval,
    test_size=N_VAL / (N_TRAIN + N_VAL),
    random_state=SEED,
    stratify=y_trainval,
)
x_test, y_test = muestrear_estratificado(x_test_full, y_test_full, N_TEST, N_CLASSES, SEED)

print(f"Entrenamiento: {x_train.shape}")
print(f"Validación:    {x_val.shape}")
print(f"Test:          {x_test.shape}")


**Hallazgos:** *(Completar: verificar que las proporciones de clases se mantienen equilibradas en los tres conjuntos tras el muestreo estratificado.)*


### 4.2 Normalización de los valores de píxel

Los valores de píxel originales están en el rango entero `[0, 255]`. Se normalizan a `[0, 1]` (`float32`) dividiendo por 255, lo que ayuda a que el descenso del gradiente converja de forma más rápida y estable — a diferencia de datos tabulares heterogéneos, para imágenes no se utiliza `StandardScaler`, ya que todos los píxeles comparten la misma escala y rango de origen.


In [ ]:
# ============================================================
# Normalización de píxeles a [0, 1]
# ============================================================
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

N_FEATURES = 32 * 32 * 3
print(f"Rango de valores tras normalizar (train): [{x_train.min():.3f}, {x_train.max():.3f}]")
print(f"Cantidad de features al aplanar cada imagen: {N_FEATURES}")


In [ ]:
# ============================================================
# Histograma de intensidad de píxeles antes/después de normalizar
# ============================================================
muestra_original = (x_train[:200] * 255).astype("uint8").flatten()
muestra_normalizada = x_train[:200].flatten()

fig_hist = make_subplots(rows=1, cols=2, subplot_titles=("Antes de normalizar (0-255)", "Después de normalizar (0-1)"))
fig_hist.add_trace(
    go.Histogram(x=muestra_original, marker_color="#E45756", name="Píxeles [0-255]",
                 hovertemplate="Valor: %{x}<br>Frecuencia: %{y}<extra></extra>"),
    row=1, col=1,
)
fig_hist.add_trace(
    go.Histogram(x=muestra_normalizada, marker_color="#4C78A8", name="Píxeles [0-1]",
                 hovertemplate="Valor: %{x}<br>Frecuencia: %{y}<extra></extra>"),
    row=1, col=2,
)
fig_hist.update_xaxes(title_text="Valor de píxel", row=1, col=1)
fig_hist.update_xaxes(title_text="Valor de píxel", row=1, col=2)
fig_hist.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig_hist.update_layout(title="Distribución de intensidad de píxeles", template="plotly_white", legend_title_text="Serie")
fig_hist.show()


**Hallazgos:** *(Completar: confirmar que el histograma normalizado conserva la misma forma que el original, solo reescalado al rango [0,1], y comentar brevemente la justificación técnica de este paso.)*


## 5. Utilidades de visualización

Antes de definir el modelo, se construyen dos funciones auxiliares que se reutilizarán en todo el notebook para graficar con **Plotly**:

1. `obtener_paso_marcadores(indice)`: determina cada cuántas épocas se dibuja un marcador sobre una curva, en función del orden en que esa curva fue agregada al gráfico. Esto evita que los gráficos con muchas curvas y muchas épocas se saturen visualmente: la **primera** curva muestra un marcador en **cada época**, la **segunda** cada **5 épocas**, la **tercera** cada **10 épocas**, y desde la cuarta en adelante se suma **+5** cada vez.
2. `graficar_comparacion_entrenamiento(...)`: dibuja, para una o más configuraciones (experimentos), las curvas de *train* (línea continua) y *validación* (línea segmentada) de una métrica (pérdida o accuracy) por época, con:
   - Etiquetas de ejes, título y leyenda.
   - Marcadores por época (según la función anterior) que al pasar el mouse muestran la época y el valor exacto.
   - Un marcador especial en el **punto de quiebre** (la época de mejor desempeño en validación), que es una aproximación visual al punto en que convendría aplicar *Early Stopping*.


In [ ]:
# ============================================================
# Utilidad 1: paso de marcadores según el orden de la curva
# ============================================================
def obtener_paso_marcadores(indice):
    """Devuelve cada cuántas épocas se dibuja un marcador, según el orden
    (índice, partiendo de 0) en que la curva fue agregada al gráfico.
    1ra curva -> cada época | 2da -> cada 5 | 3ra -> cada 10 | 4ta+ -> +5 cada vez.
    """
    if indice == 0:
        return 1
    elif indice == 1:
        return 5
    elif indice == 2:
        return 10
    else:
        return 10 + 5 * (indice - 2)


def indices_marcadores(n_epocas, paso):
    """Índices (0-based) de las épocas donde se dibuja un marcador,
    incluyendo siempre la primera y la última época."""
    idx = list(range(0, n_epocas, paso))
    if (n_epocas - 1) not in idx:
        idx.append(n_epocas - 1)
    return sorted(set(idx))


print("Funciones de utilidad de marcadores definidas.")


In [ ]:
# ============================================================
# Utilidad 2: comparación de curvas de entrenamiento (Plotly)
# ============================================================
def graficar_comparacion_entrenamiento(historias, metrica, titulo, etiqueta_y, modo="min"):
    """
    Grafica curvas de train vs. validación por época para una o más
    configuraciones (experimentos), usando Plotly.

    Parámetros
    ----------
    historias : dict {etiqueta: history.history}
        Diccionario con los resultados de entrenamiento de cada experimento.
    metrica : str
        Nombre base de la métrica en el history de Keras (ej. 'loss', 'accuracy').
    titulo : str
        Título del gráfico.
    etiqueta_y : str
        Título del eje Y.
    modo : str
        'min' si el punto de quiebre corresponde al mínimo de la curva de
        validación (ej. loss), 'max' si corresponde al máximo (ej. accuracy).
    """
    colores = px.colors.qualitative.Plotly
    fig = go.Figure()

    for i, (etiqueta, hist) in enumerate(historias.items()):
        color = colores[i % len(colores)]
        train_vals = hist[metrica]
        val_vals = hist[f"val_{metrica}"]
        n_epocas = len(train_vals)
        epocas = list(range(1, n_epocas + 1))

        paso = obtener_paso_marcadores(i)
        marker_idx = indices_marcadores(n_epocas, paso)

        # --- Curva de entrenamiento (línea continua) ---
        fig.add_trace(go.Scatter(
            x=epocas, y=train_vals,
            mode="lines+markers",
            line=dict(color=color, width=2),
            marker=dict(size=6, symbol="circle"),
            name=f"{etiqueta} (train)",
            hovertemplate="Época %{x}<br>" + etiqueta_y + ": %{y:.4f}<extra>" + etiqueta + " (train)</extra>",
        ))
        fig.data[-1].marker.size = [8 if idx in marker_idx else 0 for idx in range(n_epocas)]

        # --- Curva de validación (línea segmentada) ---
        fig.add_trace(go.Scatter(
            x=epocas, y=val_vals,
            mode="lines+markers",
            line=dict(color=color, width=2, dash="dash"),
            marker=dict(size=6, symbol="diamond"),
            name=f"{etiqueta} (val)",
            hovertemplate="Época %{x}<br>" + etiqueta_y + ": %{y:.4f}<extra>" + etiqueta + " (val)</extra>",
        ))
        fig.data[-1].marker.size = [8 if idx in marker_idx else 0 for idx in range(n_epocas)]

        # --- Punto de quiebre (mejor época según validación) ---
        if modo == "min":
            epoca_quiebre = int(np.argmin(val_vals))
        else:
            epoca_quiebre = int(np.argmax(val_vals))

        fig.add_trace(go.Scatter(
            x=[epoca_quiebre + 1], y=[val_vals[epoca_quiebre]],
            mode="markers",
            marker=dict(size=14, symbol="star", color=color, line=dict(width=1, color="black")),
            name=f"{etiqueta} (punto de quiebre)",
            hovertemplate=(
                "<b>Punto de quiebre</b><br>" + etiqueta +
                "<br>Época: %{x}<br>" + etiqueta_y + ": %{y:.4f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        title=titulo,
        xaxis_title="Época",
        yaxis_title=etiqueta_y,
        legend_title_text="Curva",
        hovermode="closest",
        template="plotly_white",
    )
    fig.show()


print("Función de comparación de entrenamiento definida.")


Con estas dos utilidades definidas, todos los gráficos de curvas de entrenamiento del resto del notebook (comparaciones de hiperparámetros, regularización, etc.) se construyen llamando a `graficar_comparacion_entrenamiento(...)`, evitando repetir código de graficación en cada experimento.


## 6. Definición de la arquitectura del modelo (MLP)

### 6.1 Función de construcción del modelo

Se define una función `construir_modelo(...)` que arma una red **feed-forward** (Perceptrón Multicapa) totalmente configurable, para poder reutilizarla en todos los experimentos de este notebook variando un único parámetro a la vez:

- `capas_ocultas`: lista/tupla con la cantidad de neuronas de cada capa oculta.
- `activacion`: función de activación de las capas ocultas (`'relu'`, `'tanh'`, `'sigmoid'`, etc.).
- `dropout_rate`: tasa de Dropout después de cada capa oculta (0 desactiva la técnica).
- `l2_lambda`: intensidad de la regularización L2 sobre los pesos (0 desactiva la técnica).
- `learning_rate`: tasa de aprendizaje del optimizador.
- `loss`: función de error a utilizar.
- `optimizer_name`: optimizador (`'adam'`, `'sgd'`, etc.).

La entrada del modelo recibe directamente cada imagen con forma `(32, 32, 3)` y una capa `Flatten` la convierte en un vector de 3.072 valores antes de las capas densas. La capa de salida usa `N_CLASSES` neuronas con activación `softmax`, adecuada para clasificación multiclase junto con `sparse_categorical_crossentropy` (las etiquetas de CIFAR-10 ya son enteros 0-9).


In [ ]:
# ============================================================
# Función de construcción del modelo MLP (reutilizable en todo el notebook)
# ============================================================
def construir_modelo(
    capas_ocultas=(128, 64),
    activacion="relu",
    dropout_rate=0.0,
    l2_lambda=0.0,
    learning_rate=0.001,
    loss="sparse_categorical_crossentropy",
    optimizer_name="adam",
):
    """Construye y compila un MLP feed-forward totalmente configurable."""
    regularizer = regularizers.l2(l2_lambda) if l2_lambda > 0 else None

    modelo = keras.Sequential()
    modelo.add(layers.Input(shape=(32, 32, 3)))
    modelo.add(layers.Flatten())  # 32x32x3 -> vector de 3072 valores

    for n_neuronas in capas_ocultas:
        modelo.add(layers.Dense(n_neuronas, activation=activacion, kernel_regularizer=regularizer))
        if dropout_rate > 0:
            modelo.add(layers.Dropout(dropout_rate))

    # Capa de salida: softmax + N_CLASSES neuronas (clasificación multiclase)
    modelo.add(layers.Dense(N_CLASSES, activation="softmax"))

    if optimizer_name == "adam":
        optimizador = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "sgd":
        optimizador = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    modelo.compile(optimizer=optimizador, loss=loss, metrics=["accuracy"])
    return modelo


modelo_ejemplo = construir_modelo()
modelo_ejemplo.summary()


**Hallazgos:** *(Completar: describir brevemente la arquitectura base elegida — cantidad de capas, neuronas por capa — y la cantidad total de parámetros entrenables que reporta `summary()`.)*


## 7. Entrenamiento del modelo base

Se entrena un primer modelo **base**, sin ninguna técnica de regularización, con hiperparámetros iniciales razonables. Este modelo servirá como punto de comparación para todos los experimentos posteriores.


In [ ]:
# ============================================================
# Configuración base de entrenamiento
# ============================================================
EPOCHS_BASE = 40
BATCH_SIZE_BASE = 64
LEARNING_RATE_BASE = 0.001

modelo_base = construir_modelo(
    capas_ocultas=(128, 64),
    activacion="relu",
    learning_rate=LEARNING_RATE_BASE,
)

historia_base = modelo_base.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_BASE,
    batch_size=BATCH_SIZE_BASE,
    verbose=2,
)


### 7.1 Curvas de entrenamiento del modelo base

A continuación se grafican, por separado, la **pérdida (loss)** y el **accuracy** de entrenamiento vs. validación por época.


In [ ]:
# ============================================================
# Curva de Loss — modelo base
# ============================================================
graficar_comparacion_entrenamiento(
    {"Modelo base": historia_base.history},
    metrica="loss",
    titulo="Modelo base — Pérdida (Loss) por época",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


In [ ]:
# ============================================================
# Curva de Accuracy — modelo base
# ============================================================
graficar_comparacion_entrenamiento(
    {"Modelo base": historia_base.history},
    metrica="accuracy",
    titulo="Modelo base — Accuracy por época",
    etiqueta_y="Accuracy",
    modo="max",
)


**Hallazgos:** *(Completar: indicar si se observan señales de overfitting — divergencia entre train y validación —, en qué época aproximada comienza, y qué tan estable es el entrenamiento.)*


## 8. Análisis del efecto de los hiperparámetros

En esta sección se realizan **experimentos controlados**, variando **un único hiperparámetro a la vez** y manteniendo el resto fijo, para poder atribuir los cambios observados en el entrenamiento a ese hiperparámetro en particular.

Para decidir cuál configuración de cada experimento es la "mejor", el criterio en todos los casos es el mismo: se comparan **accuracy y loss de validación** (nunca de entrenamiento, ya que este último puede mejorar por pura memorización) y la **brecha train-validación** como indicador de sobreajuste. La elección no se basa jamás en el conjunto de test, que permanece reservado hasta la evaluación final (sección 11).


### 8.1 Experimento: tasa de aprendizaje (learning rate)

Se entrena la misma arquitectura variando únicamente la tasa de aprendizaje del optimizador.


In [ ]:
# ============================================================
# Experimento 1: tasa de aprendizaje
# ============================================================
tasas_aprendizaje = [0.0001, 0.001, 0.01]
EPOCHS_EXPERIMENTO = 25

historias_lr = {}
resumen_lr = []

for lr in tasas_aprendizaje:
    print(f"Entrenando con learning_rate = {lr} ...")
    modelo = construir_modelo(learning_rate=lr)
    hist = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_EXPERIMENTO,
        batch_size=BATCH_SIZE_BASE,
        verbose=0,
    )
    historias_lr[f"lr={lr}"] = hist.history
    resumen_lr.append({
        "learning_rate": lr,
        "loss_final_train": hist.history["loss"][-1],
        "loss_final_val": hist.history["val_loss"][-1],
        "accuracy_final_val": hist.history["val_accuracy"][-1],
    })

print("Entrenamiento de los tres modelos finalizado.")


In [ ]:
# ============================================================
# Comparación visual — tasa de aprendizaje
# ============================================================
graficar_comparacion_entrenamiento(
    historias_lr,
    metrica="loss",
    titulo="Efecto de la tasa de aprendizaje — Loss",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


In [ ]:
# ============================================================
# Tabla resumen — tasa de aprendizaje
# ============================================================
tabla_resumen_lr = pd.DataFrame(resumen_lr)
display(tabla_resumen_lr)


**Hallazgos:** *(Completar: comparar el comportamiento con tasas de aprendizaje bajas, intermedias y altas —convergencia lenta, entrenamiento estable, u oscilaciones/divergencia—, y justificar cuál se considera más adecuada para este problema.)*


### 8.2 Experimento: tamaño de batch (batch size)

Se fija la tasa de aprendizaje elegida y se varía el tamaño de batch en **cinco** valores (potencias de 2, de 16 a 256), comparando tanto la estabilidad ("ruido") de las curvas de entrenamiento como el desempeño final en validación.


In [ ]:
# ============================================================
# Experimento 2: tamaño de batch
# ============================================================
tamanos_batch = [16, 32, 64, 128, 256]
LR_ELEGIDA = 0.001  # ajustar según el hallazgo del experimento anterior

historias_batch = {}
resumen_batch = []

for bs in tamanos_batch:
    print(f"Entrenando con batch_size = {bs} ...")
    modelo = construir_modelo(learning_rate=LR_ELEGIDA)
    hist = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_EXPERIMENTO,
        batch_size=bs,
        verbose=0,
    )
    historias_batch[f"batch={bs}"] = hist.history
    resumen_batch.append({
        "batch_size": bs,
        "loss_final_train": hist.history["loss"][-1],
        "loss_final_val": hist.history["val_loss"][-1],
        "accuracy_final_val": hist.history["val_accuracy"][-1],
    })

print("Entrenamiento de los cinco modelos finalizado.")


In [ ]:
# ============================================================
# Comparación visual — tamaño de batch
# ============================================================
graficar_comparacion_entrenamiento(
    historias_batch,
    metrica="loss",
    titulo="Efecto del tamaño de batch — Loss",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


In [ ]:
# ============================================================
# Tabla resumen — tamaño de batch
# ============================================================
tabla_resumen_batch = pd.DataFrame(resumen_batch)
display(tabla_resumen_batch)


**Hallazgos:** *(Completar: describir la diferencia en el "ruido" de las curvas entre batches pequeños y grandes, relacionarlo con el número de actualizaciones de gradiente por época, e indicar si algún tamaño de batch domina claramente en accuracy de validación.)*


### 8.3 Experimento: capacidad de la red (número de neuronas/capas)

Se comparan **cuatro** arquitecturas de complejidad creciente — pequeña, intermedia, grande y muy grande (4 capas ocultas) —, manteniendo fijos el resto de los hiperparámetros. El criterio de comparación combina accuracy de validación, brecha train-validación y cantidad de parámetros entrenables, para evaluar si el costo de una red más grande se justifica con una mejora real en generalización.


In [ ]:
# ============================================================
# Experimento 3: capacidad de la red
# ============================================================
arquitecturas = {
    "pequeña [16]": (16,),
    "intermedia [64, 32]": (64, 32),
    "grande [256, 128, 64]": (256, 128, 64),
    "muy grande [512, 256, 128, 64]": (512, 256, 128, 64),
}

historias_capacidad = {}
resumen_capacidad = []

for nombre, capas in arquitecturas.items():
    print(f"Entrenando arquitectura {nombre} ...")
    modelo = construir_modelo(capas_ocultas=capas, learning_rate=LR_ELEGIDA)
    n_parametros = modelo.count_params()
    hist = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_EXPERIMENTO,
        batch_size=BATCH_SIZE_BASE,
        verbose=0,
    )
    historias_capacidad[nombre] = hist.history
    resumen_capacidad.append({
        "arquitectura": nombre,
        "n_parametros": n_parametros,
        "accuracy_final_train": hist.history["accuracy"][-1],
        "accuracy_final_val": hist.history["val_accuracy"][-1],
        "brecha_train_val": hist.history["accuracy"][-1] - hist.history["val_accuracy"][-1],
    })

print("Entrenamiento de las cuatro arquitecturas finalizado.")


In [ ]:
# ============================================================
# Comparación visual — capacidad de la red (curvas)
# ============================================================
graficar_comparacion_entrenamiento(
    historias_capacidad,
    metrica="accuracy",
    titulo="Efecto de la capacidad de la red — Accuracy",
    etiqueta_y="Accuracy",
    modo="max",
)


In [ ]:
# ============================================================
# Tabla resumen — capacidad de la red
# ============================================================
tabla_resumen_capacidad = pd.DataFrame(resumen_capacidad)
display(tabla_resumen_capacidad)


In [ ]:
# ============================================================
# Relación entre cantidad de parámetros y accuracy de validación
# ============================================================
fig_param_vs_acc = go.Figure(
    data=[
        go.Scatter(
            x=tabla_resumen_capacidad["n_parametros"],
            y=tabla_resumen_capacidad["accuracy_final_val"],
            mode="markers+text",
            text=tabla_resumen_capacidad["arquitectura"],
            textposition="top center",
            marker=dict(size=14, color="#54A24B"),
            hovertemplate="%{text}<br>Parámetros: %{x:,}<br>Accuracy val: %{y:.4f}<extra></extra>",
            name="Arquitectura",
        )
    ]
)
fig_param_vs_acc.update_layout(
    title="Retornos decrecientes: parámetros del modelo vs. accuracy de validación",
    xaxis_title="Cantidad de parámetros entrenables",
    yaxis_title="Accuracy de validación",
    legend_title_text="Serie",
    template="plotly_white",
)
fig_param_vs_acc.show()


**Hallazgos:** *(Completar: indicar cuál arquitectura logró el menor error de entrenamiento, si esa misma arquitectura logró también el menor error de validación, si se observa overfitting creciente a medida que aumenta la capacidad de la red, y si el gráfico de parámetros vs. accuracy muestra un punto de "retornos decrecientes" a partir del cual agregar más neuronas deja de justificarse.)*


## 9. Comparación de funciones de activación

Se entrena la misma arquitectura variando únicamente la función de activación de las capas ocultas, para analizar su impacto tanto en la **velocidad de convergencia** (forma de la curva de pérdida) como en el desempeño final del modelo.


In [ ]:
# ============================================================
# Comparación de funciones de activación
# ============================================================
funciones_activacion = ["relu", "tanh", "sigmoid"]

historias_activacion = {}
resumen_activacion = []

for act in funciones_activacion:
    print(f"Entrenando con activación = {act} ...")
    modelo = construir_modelo(activacion=act, learning_rate=LR_ELEGIDA)
    hist = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_EXPERIMENTO,
        batch_size=BATCH_SIZE_BASE,
        verbose=0,
    )
    y_val_pred = np.argmax(modelo.predict(x_val, verbose=0), axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_val, y_val_pred, average="weighted", zero_division=0)

    historias_activacion[act] = hist.history
    resumen_activacion.append({
        "activacion": act,
        "accuracy_val": accuracy_score(y_val, y_val_pred),
        "precision_val": precision,
        "recall_val": recall,
        "f1_val": f1,
    })

print("Entrenamiento con las tres funciones de activación finalizado.")


In [ ]:
# ============================================================
# Comparación visual — funciones de activación
# ============================================================
graficar_comparacion_entrenamiento(
    historias_activacion,
    metrica="loss",
    titulo="Comparación de funciones de activación — Loss",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


In [ ]:
# ============================================================
# Tabla comparativa de métricas — funciones de activación
# ============================================================
tabla_activacion = pd.DataFrame(resumen_activacion)
display(tabla_activacion)


**Hallazgos:** *(Completar: comparar la velocidad de convergencia —qué tan pronunciada es la caída de la curva en las primeras épocas— y el desempeño final —accuracy, precision, recall, F1— entre `relu`, `tanh` y `sigmoid`, y justificar cuál función de activación resulta más adecuada para este problema.)*


## 10. Comparación de funciones de error (loss)

Se compara la función de error estándar para clasificación (`sparse_categorical_crossentropy`) contra una función de error inadecuada para este tipo de problema (`mean_squared_error`, MSE), con el fin de evidenciar por qué la elección de la función de error es crítica en un problema de clasificación.

**Nota técnica:** para poder usar `mean_squared_error` con Keras en un problema de clasificación es necesario codificar la salida como *one-hot* (en vez de enteros), por lo que en este experimento se ajustan las etiquetas en consecuencia.


In [ ]:
# ============================================================
# Comparación de funciones de error: crossentropy vs. MSE
# ============================================================
y_train_onehot = keras.utils.to_categorical(y_train, N_CLASSES)
y_val_onehot = keras.utils.to_categorical(y_val, N_CLASSES)

historias_loss = {}
resumen_loss = []

# --- Modelo con crossentropy (función de error recomendada) ---
modelo_ce = construir_modelo(learning_rate=LR_ELEGIDA, loss="categorical_crossentropy")
hist_ce = modelo_ce.fit(
    x_train, y_train_onehot,
    validation_data=(x_val, y_val_onehot),
    epochs=EPOCHS_EXPERIMENTO, batch_size=BATCH_SIZE_BASE, verbose=0,
)
historias_loss["categorical_crossentropy"] = hist_ce.history

# --- Modelo con MSE (función de error no recomendada para clasificación) ---
modelo_mse = construir_modelo(learning_rate=LR_ELEGIDA, loss="mean_squared_error")
hist_mse = modelo_mse.fit(
    x_train, y_train_onehot,
    validation_data=(x_val, y_val_onehot),
    epochs=EPOCHS_EXPERIMENTO, batch_size=BATCH_SIZE_BASE, verbose=0,
)
historias_loss["mean_squared_error"] = hist_mse.history

for nombre, hist in [("categorical_crossentropy", hist_ce), ("mean_squared_error", hist_mse)]:
    resumen_loss.append({
        "funcion_error": nombre,
        "accuracy_final_val": hist.history["val_accuracy"][-1],
        "loss_final_val": hist.history["val_loss"][-1],
    })

print("Entrenamiento con ambas funciones de error finalizado.")


In [ ]:
# ============================================================
# Comparación visual — funciones de error (accuracy)
# ============================================================
graficar_comparacion_entrenamiento(
    historias_loss,
    metrica="accuracy",
    titulo="Comparación de funciones de error — Accuracy",
    etiqueta_y="Accuracy",
    modo="max",
)


In [ ]:
# ============================================================
# Tabla comparativa — funciones de error
# ============================================================
tabla_loss = pd.DataFrame(resumen_loss)
display(tabla_loss)


**Hallazgos:** *(Completar: explicar por qué `categorical_crossentropy` es la función de error apropiada para clasificación —penaliza de forma más adecuada probabilidades mal calibradas— mientras que `mean_squared_error` fue diseñada para regresión y tiende a converger más lento o a un desempeño inferior en clasificación.)*


## 11. Técnicas de optimización y regularización

Se comparan cuatro configuraciones para evidenciar el efecto de cada técnica de regularización sobre la **brecha entre el error de entrenamiento y el de validación** (indicador de overfitting):

1. **Sin regularización** (baseline).
2. **Con Dropout**.
3. **Con regularización L2**.
4. **Dropout + L2 + Early Stopping** (combinación completa).

Para el modelo final se usa además el callback `EarlyStopping` de Keras, que monitorea `val_loss`, detiene el entrenamiento cuando deja de mejorar, y restaura los mejores pesos observados.


In [ ]:
# ============================================================
# Configuración de regularización
# ============================================================
DROPOUT_RATE = 0.3
L2_LAMBDA = 0.001
EPOCHS_REG = 80
PATIENCE = 8

historias_reg = {}
resumen_reg = []

configuraciones_reg = {
    "Sin regularización": dict(dropout_rate=0.0, l2_lambda=0.0),
    "Con Dropout": dict(dropout_rate=DROPOUT_RATE, l2_lambda=0.0),
    "Con L2": dict(dropout_rate=0.0, l2_lambda=L2_LAMBDA),
}

for nombre, params in configuraciones_reg.items():
    print(f"Entrenando: {nombre} ...")
    modelo = construir_modelo(learning_rate=LR_ELEGIDA, **params)
    hist = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_REG,
        batch_size=BATCH_SIZE_BASE,
        verbose=0,
    )
    historias_reg[nombre] = hist.history

print("Entrenamiento de las configuraciones sin Early Stopping finalizado.")


In [ ]:
# ============================================================
# Modelo con Dropout + L2 + Early Stopping (obligatorio)
# ============================================================
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

modelo_completo = construir_modelo(learning_rate=LR_ELEGIDA, dropout_rate=DROPOUT_RATE, l2_lambda=L2_LAMBDA)
historia_completo = modelo_completo.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_REG,
    batch_size=BATCH_SIZE_BASE,
    callbacks=[early_stopping],
    verbose=2,
)
historias_reg["Dropout + L2 + Early Stopping"] = historia_completo.history

print(f"\nEntrenamiento detenido en la época {len(historia_completo.history['loss'])} de {EPOCHS_REG} posibles.")


In [ ]:
# ============================================================
# Comparación visual de las 4 configuraciones — Loss
# ============================================================
graficar_comparacion_entrenamiento(
    historias_reg,
    metrica="loss",
    titulo="Efecto de las técnicas de regularización — Loss",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


In [ ]:
# ============================================================
# Tabla resumen — brecha train/val por configuración de regularización
# ============================================================
for nombre, hist in historias_reg.items():
    brecha = hist["accuracy"][-1] - hist["val_accuracy"][-1]
    resumen_reg.append({
        "configuracion": nombre,
        "epocas_entrenadas": len(hist["loss"]),
        "accuracy_final_train": hist["accuracy"][-1],
        "accuracy_final_val": hist["val_accuracy"][-1],
        "brecha_train_val": brecha,
    })

tabla_regularizacion = pd.DataFrame(resumen_reg)
display(tabla_regularizacion)


**Hallazgos:** *(Completar: indicar qué configuración logró la menor brecha train/validación, si el uso combinado de Dropout + L2 + Early Stopping mejoró la estabilidad del modelo respecto al baseline, y en cuántas épocas se detuvo el entrenamiento con Early Stopping.)*


## 12. Ajuste final de hiperparámetros y reentrenamiento

A partir de todos los experimentos anteriores, se selecciona la **configuración final** del modelo, justificando cada elección con base en la evidencia obtenida:

| Hiperparámetro | Valor elegido | Justificación |
|---|---|---|
| Arquitectura (capas ocultas) | *(completar)* | *(completar)* |
| Función de activación | *(completar)* | *(completar)* |
| Tasa de aprendizaje | *(completar)* | *(completar)* |
| Tamaño de batch | *(completar)* | *(completar)* |
| Dropout | *(completar)* | *(completar)* |
| L2 | *(completar)* | *(completar)* |
| Función de error | *(completar)* | *(completar)* |

Con esta configuración final se reentrena el modelo utilizando **Early Stopping**, y se conserva el modelo resultante (`modelo_final`) para la evaluación en el conjunto de test.


In [ ]:
# ============================================================
# Configuración final elegida (ajustar según los hallazgos anteriores)
# ============================================================
CONFIG_FINAL = dict(
    capas_ocultas=(128, 64),
    activacion="relu",
    dropout_rate=DROPOUT_RATE,
    l2_lambda=L2_LAMBDA,
    learning_rate=LR_ELEGIDA,
)

early_stopping_final = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1
)

modelo_final = construir_modelo(**CONFIG_FINAL)
historia_final = modelo_final.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_REG,
    batch_size=BATCH_SIZE_BASE,
    callbacks=[early_stopping_final],
    verbose=2,
)


In [ ]:
# ============================================================
# Curvas de entrenamiento — modelo final
# ============================================================
graficar_comparacion_entrenamiento(
    {"Modelo final": historia_final.history},
    metrica="loss",
    titulo="Modelo final — Loss por época",
    etiqueta_y="Loss (crossentropy)",
    modo="min",
)


**Hallazgos:** *(Completar: comparar el desempeño del modelo final contra el modelo base de la sección 7, y describir en qué mejoró.)*


## 13. Evaluación del modelo en el conjunto de test

El conjunto de **test** nunca fue utilizado durante el entrenamiento ni en la selección de hiperparámetros, por lo que las métricas obtenidas aquí constituyen la evaluación **honesta** del modelo final.


In [ ]:
# ============================================================
# Predicciones sobre el conjunto de test
# ============================================================
y_test_proba = modelo_final.predict(x_test, verbose=0)
y_test_pred = np.argmax(y_test_proba, axis=1)

accuracy_test = accuracy_score(y_test, y_test_pred)
precision_test, recall_test, f1_test, _ = precision_recall_fscore_support(
    y_test, y_test_pred, average="weighted", zero_division=0
)

print(f"Accuracy  (test): {accuracy_test:.4f}")
print(f"Precision (test): {precision_test:.4f}")
print(f"Recall    (test): {recall_test:.4f}")
print(f"F1-score  (test): {f1_test:.4f}")


### 13.1 Cuadro resumen de métricas

Se presenta un cuadro resumen con las métricas obtenidas, calculadas tanto en su versión "weighted" (global) como por clase, para poder interpretar el desempeño del modelo en cada categoría del problema.


In [ ]:
# ============================================================
# Cuadro resumen de métricas (global y por clase)
# ============================================================
reporte_dict = classification_report(
    y_test, y_test_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
tabla_metricas = pd.DataFrame(reporte_dict).transpose()
display(tabla_metricas)


**Hallazgos:** *(Completar: interpretar las métricas obtenidas — ¿el modelo tiene un buen desempeño general?, ¿hay alguna clase con precision o recall notablemente más bajo?, ¿coincide con las confusiones esperables en CIFAR-10 —p. ej. gato/perro, auto/camión, pájaro/avión—?)*


### 13.2 Matriz de confusión

La matriz de confusión permite visualizar en detalle qué clases se confunden entre sí, información que no se aprecia directamente en las métricas agregadas. En CIFAR-10 es habitual observar confusión entre pares de clases visualmente similares (por ejemplo gato/perro o automóvil/camión).


In [ ]:
# ============================================================
# Matriz de confusión (Plotly)
# ============================================================
matriz_confusion = confusion_matrix(y_test, y_test_pred)

fig_confusion = go.Figure(
    data=go.Heatmap(
        z=matriz_confusion,
        x=CLASS_NAMES,
        y=CLASS_NAMES,
        colorscale="Blues",
        text=matriz_confusion,
        texttemplate="%{text}",
        hovertemplate="Real: %{y}<br>Predicho: %{x}<br>Cantidad: %{z}<extra></extra>",
        colorbar=dict(title="Cantidad"),
    )
)
fig_confusion.update_layout(
    title="Matriz de confusión — conjunto de test",
    xaxis_title="Clase predicha",
    yaxis_title="Clase real",
    template="plotly_white",
)
fig_confusion.show()


**Hallazgos:** *(Completar: identificar los pares de clases con mayor confusión y proponer una hipótesis de por qué ocurre — por ejemplo, similitud de forma/color entre esas clases, o la pérdida de estructura espacial al aplanar la imagen para un MLP.)*


## 14. Comparación de configuraciones — tabla resumen general

Se consolida en una única tabla el desempeño en validación de **todas** las configuraciones probadas a lo largo del notebook, para facilitar la comparación global y respaldar la elección de la configuración final.


In [ ]:
# ============================================================
# Tabla resumen general de todos los experimentos
# ============================================================
resumen_general = []

for nombre, hist in historias_lr.items():
    resumen_general.append({"experimento": f"LR — {nombre}", "accuracy_val": hist["val_accuracy"][-1], "loss_val": hist["val_loss"][-1]})
for nombre, hist in historias_batch.items():
    resumen_general.append({"experimento": f"Batch — {nombre}", "accuracy_val": hist["val_accuracy"][-1], "loss_val": hist["val_loss"][-1]})
for nombre, hist in historias_capacidad.items():
    resumen_general.append({"experimento": f"Arquitectura — {nombre}", "accuracy_val": hist["val_accuracy"][-1], "loss_val": hist["val_loss"][-1]})
for nombre, hist in historias_activacion.items():
    resumen_general.append({"experimento": f"Activación — {nombre}", "accuracy_val": hist["val_accuracy"][-1], "loss_val": hist["val_loss"][-1]})
for nombre, hist in historias_reg.items():
    resumen_general.append({"experimento": f"Regularización — {nombre}", "accuracy_val": hist["val_accuracy"][-1], "loss_val": hist["val_loss"][-1]})
resumen_general.append({"experimento": "Modelo final (test)", "accuracy_val": accuracy_test, "loss_val": None})

tabla_resumen_general = pd.DataFrame(resumen_general)
display(tabla_resumen_general)


In [ ]:
# ============================================================
# Gráfico comparativo final — accuracy de validación por experimento
# ============================================================
fig_resumen = go.Figure(
    data=[
        go.Bar(
            x=tabla_resumen_general["experimento"],
            y=tabla_resumen_general["accuracy_val"],
            marker_color="#54A24B",
            hovertemplate="%{x}<br>Accuracy: %{y:.4f}<extra></extra>",
            name="Accuracy de validación",
        )
    ]
)
fig_resumen.update_layout(
    title="Comparación general de accuracy de validación por experimento",
    xaxis_title="Experimento",
    yaxis_title="Accuracy (validación)",
    xaxis_tickangle=-45,
    legend_title_text="Métrica",
    template="plotly_white",
)
fig_resumen.show()


**Hallazgos:** *(Completar: identificar cuál fue, en términos generales, la configuración con mejor desempeño y contrastarla con la configuración final elegida en la sección 12.)*


## 15. Conclusiones generales

*(Completar con un análisis propio, respondiendo al menos:)*

1. **Carga y preprocesamiento**: ¿qué decisiones fueron más relevantes y por qué (tamaño del subconjunto, normalización)?
2. **Hiperparámetros de entrenamiento**: ¿qué tasa de aprendizaje, batch size y arquitectura resultaron más adecuados, y qué evidencia lo respalda?
3. **Funciones de activación y error**: ¿cuáles se seleccionaron finalmente y por qué?
4. **Regularización**: ¿qué técnicas ayudaron a reducir el overfitting y en qué magnitud?
5. **Métricas de evaluación**: ¿qué tan bueno es el modelo final según accuracy, precision, recall y F1-score? ¿Hay clases problemáticas?
6. **Limitación del enfoque MLP**: ¿qué techo de desempeño impone usar una red totalmente conectada sobre imágenes aplanadas, en comparación con lo que se esperaría de una CNN?
7. **Posibles mejoras futuras**: ¿qué se probaría con más tiempo/recursos (más datos, redes convolucionales, data augmentation, otros optimizadores, etc.)?


## 16. Referencias

- Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images*. Technical report, University of Toronto. Disponible (enlace al reporte técnico, al final de la página) en: [https://cave.cs.toronto.edu/kriz/cifar.html](https://cave.cs.toronto.edu/kriz/cifar.html)

## 17. Anexo — Herramientas y tecnologías de apoyo utilizadas

En el desarrollo de este trabajo se utilizaron las siguientes herramientas y tecnologías:

**Entornos y asistentes:**
- Google Colab — entorno de ejecución del notebook.
- Google Docs — redacción y organización de apuntes complementarios.
- Claude (Sonnet 5), Anthropic — apoyo en la estructuración del notebook y redacción de explicaciones.
- Gemini 3.5 Flash-Lite, Google — apoyo puntual de consulta.

**Librerías y frameworks (sitios oficiales):**
- TensorFlow — [https://www.tensorflow.org/](https://www.tensorflow.org/)
- Keras — [https://keras.io/](https://keras.io/)
- scikit-learn — [https://scikit-learn.org/](https://scikit-learn.org/)
- NumPy — [https://numpy.org/](https://numpy.org/)
- Pandas — [https://pandas.pydata.org/](https://pandas.pydata.org/)
- Plotly (Python) — [https://plotly.com/python/](https://plotly.com/python/)

*(Completar/ajustar esta lista según las herramientas efectivamente utilizadas por el grupo.)*
